In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from pathlib import Path

# === Paths ===
MODEL_DIR = "E:/llm-fine-tuning/falcon1b-lora-finetuned"  # Your fine-tuned LoRA model directory
CONVERSATION_FILE = "E:/llm-fine-tuning/input_conversation.txt"  # Path to your input conversation txt file

# === Load tokenizer & base model ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-rw-1b",
    trust_remote_code=True,
    device_map="auto",
    load_in_4bit=True,  # Remove or set to False if not using 4bit quant
    torch_dtype=torch.float16
)

# === Load LoRA adapter ===
model = PeftModel.from_pretrained(model, MODEL_DIR, device_map="auto")

model.eval()

def generate_summary(conversation_text: str, max_length=512):
    prompt = f"以下の日本語の医療会話を読み、指定された形式で要約してください：\n\n会話:\n{conversation_text}\n\n要約:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.5,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip prompt from the output to only get summary
    return summary[len(prompt):].strip()

if __name__ == "__main__":
    # Read conversation text from file
    path = Path("D:/Medbank/SOAP/original_transcript.txt")
    if not path.exists():
        raise FileNotFoundError(f"Conversation file not found: {CONVERSATION_FILE}")

    conversation_text = path.read_text(encoding="utf-8").strip()

    print("=== Conversation ===")
    print(conversation_text)
    print("\n=== Generated Summary ===")
    print(generate_summary(conversation_text))


d:\Medbank\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


d:\Medbank\myenv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kusha\.cache\huggingface\hub\models--tiiuae--falcon-rw-1b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
A new version of the following files was downloaded from https://huggingface.co/tiiuae/falcon-rw-1b:
- configuration_falcon.py
. Mak

=== Conversation ===
よっぽいしゅん こんにちは 佐藤さんどうしたの今日 今日ですか ちょっとね腰の調子がやっぱり悪いって 聞いたんだけど お盆も近いからね 墓参りでも行くどうするの今度 いやー墓参りはねあれだよ あのースーパーのお花はいつもこうて んで行くんだけど最近あれじゃね スーパーも卵とかも全部高くなってから ほんまに ほんまにあのー いやー最近しんどいわ おばあさんも一緒に墓参り行くの おばあちゃんはおばあさんはもう 今もうお家テレビばっかり見るわけ テレビ見てるわけ ほんの一人で行かないかも そう一人でね行かないといけないんだけど 息子がね最近あのー 仕事が忙しくなって 息子もあれじゃけんの 最近広島帰って渋谷って最近来たらんけん いっぱいあるね いやーねえそうやっていいんじゃけどね だけど最近暑いから熱中症にならんように気を付けて ちゃんと水分飲まないと ありがとうございます でもあれ本当にやっぱりあれじゃね お水もなかなか喉が渇くから飲めん そうだよな だけどあれは一応飲んどかないと ほんま倒れてしまうから おばあちゃんのためにもちゃんと じいさんがしっかりしとかなあかんな でもおばあちゃんいつもあの テレビばっかり見てるから テレビばっかり見とるから うん じゃあ今日は腰痛いんやったかな えっとあれなんじゃった そうそうそうそう 右の腰のところがちょっと痛いと思ってから来たんじゃけど ほんなら痛めるだけ遠くから あとシップをどうやら出しといたほうがいいか シップはね あるかお家に シップそうねシップあればええかなと思って 痛めたような腰症はいらんかい 駐車はねもう身体験 ああいらんか ほんならシップだけ出しとくから もらって帰って気を付けてね 行ってきて ありがとう ほんならねお出しに来てください

=== Generated Summary ===


d:\Medbank\myenv\Lib\site-packages\bitsandbytes\nn\modules.py:463: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


会話:

よっぽいしゅん こんにちは 佐藤さんどうしたの今日 今日ですか ちょっとね腰の調子がやっぱり悪いって 聞いたんだけど お盆も近いからね スーパーのお花はいつもこうて んで行くんだけど 最近あれじゃね スーパーも卵とかも全部高くなってから ほんまにあのー いやー最近しんどいわ あのースーパーも卵とかも全部高くなってから ほんまにあのー いやー最近しんどいわ あのースーパーも卵とかも全部高くなってから ほんまにあのー いやー最近しんどいわ あのースーパーも卵とかも全部高くなってから ほんまにあのー いやー最近しんどいわ あのースーパーも卵とかも全部高くなってから ほんまにあのー いやー最近しんどいわ あのースーパーも卵とかも全部高くなってから ほんまにあのー いやー最近しんどいわ あのースーパーも卵とかも全部高くなってから ほんまにあのー いやー最近しんどいわ あのースーパーも卵とかも全部高く
